# 02 · Middleware: hooks around the agent loop
**The limitation this fixes:** in 01 every user gets the same prompt and every tool, and there's nowhere to put a rule that must run *every* time.
A prompt is a request. **Middleware is code that always runs.**

| Need | Framework piece |
|---|---|
| Run code before / after every model call | `@before_model`, `@after_model` |
| Wrap every tool call (audit log) | `@wrap_tool_call` |
| A tool that changes the agent's state | tool returns `Command(update=...)` + `state_schema` |
| Tools and model by role and identity | `@wrap_model_call` |
| Prompt by role and language | `@dynamic_prompt` |
| Ready-made: redaction, long chats, outages, cost, flaky tools | `PIIMiddleware`, `SummarizationMiddleware`, `ModelFallbackMiddleware`, `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware`, `ToolRetryMiddleware` |

After this notebook, **03 (safety)** and **04 (RAG)** only *use* these hooks. No new syntax there, just policy.

In [ ]:
# Setup: the same cell at the top of every notebook
from dotenv import load_dotenv
load_dotenv("../.env", override=True)   # OPENAI_API_KEY + LANGSMITH_* (set LANGSMITH_TRACING=true to see traces)

MODEL = "openai:gpt-4o-mini"           # swap the provider here, e.g. "anthropic:claude-sonnet-4-6"

## What we built so far
`CareContext` and the care tools from 01, plus three plain booking tools (written exactly like the tools in 01).

In [ ]:
from common import (CareContext, SYSTEM_PROMPT, PATIENTS, FALLBACK_MODEL,
                    get_care_plan, log_symptom, get_recent_readings,
                    list_free_slots, book_appointment, patients_needing_attention,   # plain tools, like in 01
                    ask, show_state)
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

## What is middleware?
`create_agent` runs a loop: **model → tools → model → … → reply**. Middleware is code you plug into fixed points of that loop.

```
 user message
      │
      ▼
 ┌─► before_model ─────── runs before every model call. Can answer itself and skip the model
 │      │
 │   wrap_model_call ──── wraps the model call: change the tools, the model or the prompt
 │      │  🧠 model
 │   after_model ──────── sees what the model decided (HumanInTheLoopMiddleware pauses here)
 │      │
 │   wrap_tool_call ───── wraps every tool call: log it, retry it, block it
 │      │  🔧 tool
 └──────┘   loops until the model replies without calling a tool
      │
      ▼
    reply
```

You already used one in 00: `HumanInTheLoopMiddleware` is an `after_model` hook that pauses before a tool runs.

## 1. `@before_model` and `@after_model`: run code on every turn
Each hook gets the agent's `state` (the messages so far) and the `runtime` (the context from 01). Returning `None` changes nothing.

In [ ]:
from langchain.agents.middleware import before_model, after_model

@before_model
def log_before(state, runtime):
    print(f"   ➡️  before_model: {len(state['messages'])} message(s) so far, user = {runtime.context.user_id}")
    return None   # None = change nothing

@after_model
def log_after(state, runtime):
    last = state["messages"][-1]   # what the model just produced
    tool_names = []
    for call in last.tool_calls:
        tool_names.append(call["name"])
    if tool_names:
        print(f"   ⬅️  after_model: the model wants to call {tool_names}")
    else:
        print("   ⬅️  after_model: the model wrote the reply")
    return None

watched = create_agent(MODEL, tools=[get_care_plan, get_recent_readings], system_prompt=SYSTEM_PROMPT,
                       context_schema=CareContext, middleware=[log_before, log_after])

ask(watched, "What's my blood pressure target?")

**A hook can also take over.** Return a message plus `"jump_to": "end"` and the model is skipped entirely.
A toy example (03 uses exactly this pattern to block dose questions):

In [ ]:
from langchain.messages import AIMessage, HumanMessage

@before_model(can_jump_to=["end"])   # declare where this hook is allowed to jump
def ping_pong(state, runtime):
    last = state["messages"][-1]
    if isinstance(last, HumanMessage) and last.content.strip().lower() == "ping":
        return {"messages": [AIMessage("pong 🏓 (answered by middleware, no model call)")], "jump_to": "end"}
    return None

toy = create_agent(MODEL, tools=[], system_prompt=SYSTEM_PROMPT, context_schema=CareContext,
                   checkpointer=InMemorySaver(), middleware=[ping_pong])

ask(toy, "ping", thread="toy")
show_state(toy, thread="toy")   # 🧾 only 2 messages: the model was never called

## 2. `@wrap_tool_call`: an audit log around every tool
`wrap_*` hooks get a `request` and a `handler`. Calling `handler(request)` runs the real thing; your code goes before and after it.
In healthcare we must record who touched which patient's data, and when.

In [ ]:
from datetime import datetime
from langchain.agents.middleware import wrap_tool_call

AUDIT_LOG = []   # CUSTOM: in production, an append-only store

@wrap_tool_call
def audit_log(request, handler):
    ctx = request.runtime.context
    AUDIT_LOG.append({"time": datetime.now().strftime("%H:%M:%S"), "user": ctx.user_id,
                      "patient": ctx.patient_token, "tool": request.tool_call["name"]})
    return handler(request)   # run the actual tool

audited = create_agent(MODEL, tools=[get_care_plan, get_recent_readings], system_prompt=SYSTEM_PROMPT,
                       context_schema=CareContext, middleware=[audit_log])

ask(audited, "Show me my care plan and my recent readings")
AUDIT_LOG   # 🧾 receipt: one row per tool call

## 3. A tool that updates state: `verify_identity`
Tools usually return text. Returning a `Command` lets a tool also write to the agent's **state**.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def verify_identity(date_of_birth: str, runtime: ToolRuntime[CareContext]) -> Command:
    """Verify the patient's identity with their date of birth (YYYY-MM-DD). Required before booking."""
    ok = PATIENTS[runtime.context.patient_token]["date_of_birth"] == date_of_birth
    reply = "Identity verified." if ok else "That date of birth doesn't match our records."
    return Command(update={"identity_verified": ok,
                           "messages": [ToolMessage(reply, tool_call_id=runtime.tool_call_id)]})

## 4. Add our own field to the agent's state

In [ ]:
from typing import NotRequired
from langchain.agents import AgentState

class CareState(AgentState):          # messages + our flag
    identity_verified: NotRequired[bool]

## 5. `@wrap_model_call`: rebuild the tool list (and model) on every call
Patients never see doctor tools. Booking tools only appear after identity is verified. Doctors get a stronger model.

In [ ]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

SHOW_TOOLS = True                                          # workshop receipt: print the tools each call sees
DOCTOR_ONLY = {"patients_needing_attention"}               # CUSTOM: access policy
NEEDS_IDENTITY = {"list_free_slots", "book_appointment"}   # CUSTOM: access policy
strong_model = init_chat_model(FALLBACK_MODEL)

@wrap_model_call(state_schema=CareState)
def pick_tools_and_model(request, handler):
    role = request.runtime.context.role
    tools = request.tools
    if role != "doctor":
        tools = [t for t in tools if t.name not in DOCTOR_ONLY]
    if not request.state.get("identity_verified"):
        tools = [t for t in tools if t.name not in NEEDS_IDENTITY]
    if SHOW_TOOLS:
        print("   🔧 tools this call:", [t.name for t in tools])
    model = strong_model if role == "doctor" else request.model
    return handler(request.override(tools=tools, model=model))

## 6. `@dynamic_prompt`: the system prompt is written fresh on every call
Same agent, different person: a doctor gets a brief, clinical tone; a patient gets plain words, in their language.

In [ ]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def care_prompt(request):
    ctx = request.runtime.context
    if ctx.role == "doctor":
        tone = "You are talking to a doctor. Be brief and clinical."
    else:
        tone = "You are talking to a patient. Use plain, warm language."
    return (f"{SYSTEM_PROMPT}\n- Before booking, ask for date of birth and call verify_identity.\n"
            f"{tone}\nAlways reply in {ctx.language}.")

## 7. Prebuilt middleware: configure it, don't write it
LangChain ships the common ones. We only supply settings.
`PIIMiddleware` needs to know *our* identifier formats; the redacting itself is LangChain's.
It cleans the user's input, the model's output **and tool results** (that matters in 04, when we retrieve documents).

In [ ]:
from langchain.agents.middleware import PIIMiddleware

pii = [
    PIIMiddleware("mrn", detector=r"MRN-\d{6}", strategy="redact",          # CUSTOM: our MRN format
                  apply_to_input=True, apply_to_output=True, apply_to_tool_results=True),
    PIIMiddleware("phone", detector=r"\b555-\d{4}\b", strategy="redact",    # CUSTOM: our phone format
                  apply_to_input=True, apply_to_output=True, apply_to_tool_results=True),
]

## Putting it together: the whole stack is one list
Almost every line is a LangChain import. **Order matters:** it runs top to bottom on the way in, like layers of an onion.
Redaction sits above summarization, so identifiers never reach a summary.

In [ ]:
from langchain.agents.middleware import (
    ModelCallLimitMiddleware, ModelFallbackMiddleware,
    SummarizationMiddleware, ToolCallLimitMiddleware, ToolRetryMiddleware)

middleware = [
    *pii,                                                         # redact MRNs and phones
    SummarizationMiddleware(MODEL, trigger=("messages", 40), keep=("messages", 20)),   # long chats
    care_prompt,                                                  # prompt by role and language
    pick_tools_and_model,                                         # tools by role and identity
    ModelFallbackMiddleware(FALLBACK_MODEL),                      # model down? try another
    ModelCallLimitMiddleware(run_limit=10),                       # cost guard
    ToolCallLimitMiddleware(run_limit=10),                        # cost guard
    ToolRetryMiddleware(max_retries=2),                           # flaky tools
    audit_log,                                                    # who touched what
]

tools = [get_care_plan, log_symptom, get_recent_readings,
         verify_identity, list_free_slots, book_appointment, patients_needing_attention]
agent = create_agent(MODEL, tools=tools, context_schema=CareContext,
                     checkpointer=InMemorySaver(), middleware=middleware)

## Demo: booking tools appear only after verification
Watch the 🔧 line grow.

In [ ]:
ask(agent, "I'd like to book an appointment with my doctor", thread="book")
ask(agent, "1970-03-14", thread="book")

In [ ]:
ask(agent, "The first slot please", thread="book")
agent.get_state({"configurable": {"thread_id": "book"}}).values["identity_verified"]   # 🧾 receipt: our state field

## ❌ Failure case: wrong date of birth, no booking tools

In [ ]:
ask(agent, "Book me an appointment. My date of birth is 1999-01-01", thread="impostor")

## Same agent, doctor context: different tools, model and tone

In [ ]:
dr_lee = CareContext(user_id="dr_lee", role="doctor")
ask(agent, "Who needs attention today?", thread="doctor", context=dr_lee)
ask(agent, "Who needs attention today?", thread="patient-tries")   # patient: tool isn't even offered

In [ ]:
ask(agent, "What's my blood pressure target?", thread="es", context=CareContext(language="Spanish"))

## PII: the MRN never reaches the model

In [ ]:
ask(agent, "My MRN is MRN-100001 and my phone is 555-0142. What's my care plan?", thread="pii")
show_state(agent, thread="pii")   # 🧾 the saved message says [REDACTED_MRN] and [REDACTED_PHONE]

🧾 **Receipt:** the audit log saw every tool call in this notebook.

In [ ]:
AUDIT_LOG

## Where this lives in the backend
- `app/middleware.py`: `build_middleware()` (the one list), `pick_tools_and_model`, `care_prompt`, `pii_middleware()`, `audit_log`
- `app/tools.py`: `verify_identity`, booking tools

Same primitives, just split into files. The notebook never imports `app/`.